In [ ]:
# %% [markdown]
# # 06 – Ablation Hyperparameter Sweep (always synthetic for controlled study)

# %%
import itertools
import numpy as np
from memorypoison_audit.core.agent_orchestrator import AgentOrchestrator
from memorypoison_audit.mitigation.sanitization_hooks import SanitizationHooks
from memorypoison_audit.attacks.gradient_free_perturber import GradientFreePerturber
from memorypoison_audit.benchmarks.metrics import MetricsCalculator
from memorypoison_audit.data_loader import SyntheticDataGenerator
from shared_functions import load_configs, set_seed, save_metrics

attack_cfg, _ = load_configs()
set_seed(42)
param_grid = {
    'perturbation_budget': [0.01, 0.05, 0.1],
    'lof_contamination': [0.1, 0.2, 0.3]
}
results = []
for budget, contam in itertools.product(param_grid['perturbation_budget'], param_grid['lof_contamination']):
    session_id = f"sweep_{budget}_{contam}"
    hooks = SanitizationHooks(enabled=True, config={'method': 'lof', 'contamination': contam, 'n_neighbors': 20})
    agent = AgentOrchestrator(session_id, attack_cfg, sanitization_hooks=hooks)
    # Populate with synthetic facts
    gen = SyntheticDataGenerator()
    facts = gen.generate_facts(num_facts=200)
    for fact in facts:
        agent.memory_store.add_fact(session_id, fact)
    # Inject with given budget
    perturber = GradientFreePerturber(budget=budget)
    for text in attack_cfg['attack']['malicious_texts']:
        perturber.apply_to_memory(agent.memory_store, session_id, text)
    # Evaluate ASR
    keywords = ["COMPROMISED", "ADMIN123"]
    asr_vals = []
    for turn in range(1, 11):
        agent.generate_response(f"Q{turn}")
        retrieved = agent.memory_store.query(session_id, f"Q{turn}", top_k=3)
        texts = [r['text'] for r in retrieved]
        asr = MetricsCalculator.attack_success_rate(texts, keywords)
        asr_vals.append(asr)
    avg_asr = np.mean(asr_vals)
    results.append((budget, contam, avg_asr))
    print(f"budget={budget}, contam={contam} -> ASR={avg_asr:.3f}")

# %%
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.DataFrame(results, columns=['budget', 'contam', 'ASR'])
pivot = df.pivot(index='budget', columns='contam', values='ASR')
plt.figure(figsize=(8,6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='coolwarm')
plt.title('Ablation: ASR vs Perturbation Budget and Contamination (Synthetic)')
plt.show()

save_metrics("ablation", {"results": results}, data_source="synthetic")